# Camada Silver: Limpeza e Tratamento de Pedidos
**Squad 2 | Projeto: Merca Data Platform**

> Este notebook é o núcleo de qualidade de dados (*Data Quality*) do nosso pipeline. A sua função é ler os micro-lotes da Camada Bronze, aplicar o Contrato de Dados (Regras de Negócio) e persistir os registos limpos e validados na Camada Silver.

**Regras de Negócio Aplicadas nesta Etapa:**
* **Regra 1:** Verificação de integridade (chaves não nulas) e remoção de duplicados exatos.
* **Regra 3:** Valores financeiros estritamente positivos (`valor_total` e `valor_frete` >= 0).
* **Regra 4:** Validação de formato de datas (Timestamp ISO 8601).
* **Regra 5:** Filtragem rigorosa por `status_pedido` autorizado.
* **Regra 10:** Coerência temporal (`dt_ultima_atualizacao_status` >= `dt_pedido`).

In [0]:
# Garante que a biblioteca nativa do Delta está instalada no cluster (necessária para a gravação da Silver)
%pip install deltalake

# 1. Configuração e Motor Utilitário
**Carga de Dependências e Mapeamento da Tabela**

> Injetamos as funções globais da Squad (como gestão de acessos e leitura de checkpoints) e preparamos as funções do PySpark que serão o nosso "motor de regras" para filtrar e transformar os dados.

In [0]:
%run ../99_utils/feat_squad2_99_helpers

In [0]:
# 2. Importa as funções do PySpark essenciais para a limpeza e validação de dados
from pyspark.sql.functions import col, lit, current_timestamp, to_timestamp, trim, upper

# 3. Define a tabela alvo deste pipeline
TABELA = "ecommerce_pedidos"

# 4. Lista oficial de Status permitidos (Regra 5 do Contrato de Dados)
STATUS_PERMITIDOS = [
    "Processando", 
    "Pagamento Aprovado", 
    "Separacao", 
    "Em Transporte", 
    "Entregue", 
    "Cancelado"
]

log.info(f"✅ Ambiente Silver configurado com sucesso para a tabela: {TABELA}")

# 2. Motor de Leitura Incremental
**Identificação e Carga do Micro-lote**

> Garantia do Princípio de Idempotência (Regra 2) na Camada Silver. Comparamos os `_snapshot_id` disponíveis na Bronze com o log de controle da Silver (`processed_snapshots.json`). Apenas dados virgens são carregados na memória para tratamento, evitando duplicidade e otimizando o processamento computacional.

In [0]:
try:
    log.info("🔍 Iniciando varredura de pacotes entre Bronze e Silver...")
    
    # 1. Carrega a tabela Bronze completa (Apenas referência, sem processamento pesado)
    df_bronze_full = ler_delta("bronze", TABELA)
    
    # 2. Extrai a lista de snapshots únicos que já existem lá na Bronze
    # Usamos list comprehension e collect() para trazer isso para uma lista Python
    snapshots_bronze = [row["_snapshot_id"] for row in df_bronze_full.select("_snapshot_id").distinct().collect()]
    
    # 3. Lê o arquivo de controle para saber o que a Silver JÁ processou no passado
    processados_silver = ler_checkpoint("silver", TABELA)
    
    # 4. A mágica da matemática de conjuntos: Filtra apenas os IDs que estão na Bronze, mas não na Silver
    novos_snapshots = sorted([s for s in snapshots_bronze if s not in processados_silver])
    
    # 5. Carrega o micro-lote para a memória ou encerra se não houver novidades
    if not novos_snapshots:
        log.info("✅ Nenhum snapshot novo na Bronze para processar na Silver!")
        df_lote = None
    else:
        log.info(f"📦 Encontrados {len(novos_snapshots)} snapshot(s) novo(s) para tratamento.")
        
        # Filtra o DataFrame da Bronze para trazer APENAS as linhas dos pacotes novos
        df_lote = df_bronze_full.filter(col("_snapshot_id").isin(novos_snapshots))
        
        qtd_linhas_lote = df_lote.count()
        log.info(f"🚀 Micro-lote isolado com sucesso: {qtd_linhas_lote} linhas carregadas para validação.")

except Exception as e:
    log.error(f"❌ Falha ao carregar o micro-lote da Bronze: {str(e)}")
    raise

# 3. Limpeza e Tratamento (Data Quality)
**Aplicação do Contrato de Dados com Tabela de Quarentena**

> Nesta etapa, o micro-lote passa por uma "alfândega" rigorosa de validação. Registros aprovados seguem para a Camada Silver. Já os registros que violam as regras de negócio (Contrato de Dados) não são descartados: eles são separados e redirecionados para uma tabela de **Quarentena** para futura auditoria e correção pelas áreas de negócio, garantindo rastreabilidade total da informação.

In [0]:
try:
    if df_lote:
        log.info("🧹 Iniciando motor de qualidade de dados (Limpeza e Quarentena)...")
        
        # 1. Remove duplicatas exatas logo de cara (isso é lixo puro, nem vai para quarentena)
        df_base = df_lote.dropDuplicates()
        
        # 2. Define a regra de ouro do Contrato de Dados
        contrato_de_dados = (
            col("id_pedido").isNotNull() & 
            col("id_cliente").isNotNull() &
            (col("valor_total") >= 0) & 
            (col("valor_frete") >= 0) &
            col("status_pedido").isin(STATUS_PERMITIDOS) &
            col("dt_pedido").isNotNull() & 
            col("dt_ultima_atualizacao_status").isNotNull() &
            (col("dt_ultima_atualizacao_status") >= col("dt_pedido"))
        )
        
        # 3. Bifurcação dos dados (Aprovados vs Quarentena)
        df_silver = df_base.filter(contrato_de_dados)
        df_quarentena = df_base.filter(~contrato_de_dados) # O '~' inverte a regra (pega os que falharam)
        
        # Auditoria rápida
        linhas_antes = df_lote.count()
        linhas_silver = df_silver.count()
        linhas_quarentena = df_quarentena.count()
        
        log.info(f"✨ Triagem concluída! Originais: {linhas_antes} | Aprovados: {linhas_silver} | Quarentena: {linhas_quarentena}")
        
    else:
        df_silver = None
        df_quarentena = None
        log.info("⏩ Nenhum dado para limpar nesta execução.")

except Exception as e:
    log.error(f"❌ Falha durante a triagem do micro-lote: {str(e)}")
    raise

# 4. Auditoria e Persistência
**Gravação Bifurcada: Camada Silver e Quarentena**

> Adição dos metadados finais de rastreabilidade e persistência em formato Delta. 
* Os dados limpos recebem a tag `_camada = silver` e são salvos na estrutura definitiva. 
* Os dados reprovados recebem a tag `_camada = quarantine` e são isolados na respectiva pasta. 
Por fim, o checkpoint (`.json`) é atualizado para garantir que este lote não será processado novamente.

In [0]:
try:
    if df_silver:
        log.info("🏷️ Adicionando metadados de auditoria...")
        
        # --- 1. GRAVAÇÃO DOS DADOS APROVADOS (SILVER) ---
        df_final_silver = df_silver \
            .withColumn("silver_processed_at", current_timestamp()) \
            .withColumn("_camada", lit("silver"))
        
        log.info("💾 Iniciando gravação na Silver...")
        sucesso_silver = gravar_delta(df_final_silver, "silver", TABELA, mode="append")
        
        # --- 2. GRAVAÇÃO DOS DADOS REPROVADOS (QUARENTENA) ---
        sucesso_quarentena = True
        if df_quarentena and df_quarentena.count() > 0:
            log.info(f"⚠️ Enviando {df_quarentena.count()} registro(s) para a Quarentena...")
            df_final_quarentena = df_quarentena \
                .withColumn("quarentena_processed_at", current_timestamp()) \
                .withColumn("_camada", lit("quarantine"))
            
            # Grava na pasta squad2/quarantine/ecommerce_pedidos
            sucesso_quarentena = gravar_delta(df_final_quarentena, "quarantine", TABELA, mode="append")
        
        # --- 3. ATUALIZAÇÃO DO CHECKPOINT JSON ---
        if sucesso_silver and sucesso_quarentena:
            processados_silver.update(novos_snapshots)
            salvar_checkpoint("silver", TABELA, processados_silver)
            log.info(f"✅ Gravação concluída e checkpoint JSON atualizado para {len(processados_silver)} pacote(s).")
            
    else:
         log.info("⏩ Sem dados para gravar.")
            
except Exception as e:
    log.error(f"❌ Erro na gravação: {str(e)}")
    raise

# 5. Validação de Saúde (Sanity Check)
**Inspeção Final do Formato e Volumetria**

In [0]:
from deltalake import DeltaTable

try:
    log.info(f"🧪 Iniciando testes de validação para a tabela Silver: {TABELA}")
    
    df_validacao = ler_delta("silver", TABELA)
    total_registros = df_validacao.count()
    
    # Extrai metadados do Delta
    caminho_tabela = get_delta_path("silver", TABELA)
    storage_opts = get_storage_options()
    dt = DeltaTable(caminho_tabela, storage_options=storage_opts)
    particoes = dt.metadata().partition_columns
    particoes_str = ", ".join(particoes) if particoes else "Nenhuma (Tabela Flat)"
    
    print(f"📊 RELATÓRIO DE VALIDAÇÃO SILVER — SQUAD 2")
    print(f"{'-'*55}")
    print(f"✅ Status da Tabela   : DISPONÍVEL")
    print(f"📌 Caminho Físico     : {caminho_tabela}")
    print(f"🔢 Total de Registros : {total_registros} linhas consolidadas")
    print(f"🗂️  Particionamento    : {particoes_str}")
    print(f"{'-'*55}\n")
    
    print("📋 Esquema Estrutural de Metadados:")
    df_validacao.printSchema()
    
    print("\n👀 Amostra dos Primeiros Registros Tratados:")
    display(df_validacao.limit(10))

except Exception as e:
    log.error(f"❌ Falha crítica na validação da Camada Silver: {str(e)}")
    raise